In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import copy

# 1. Device Selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Reading and Filtering (The step that resolves the performance warning)
print("Loading data...")
df_raw = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
base_features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']

# We only take the columns we need and copy them (Prevents fragmentation error)
data = df_raw[base_features].copy().dropna()

# 3. Time Encoding (Sin/Cos)
print("Time Coding (Sin/Cos) is applied...")
# We collect the new columns in a separate DataFrame and combine them with pd.concat
time_features = pd.DataFrame(index=data.index)
time_features['hour'] = time_features.index.hour
time_features['month'] = time_features.index.month

time_features['hour_sin'] = np.sin(2 * np.pi * time_features['hour'] / 24.0)
time_features['hour_cos'] = np.cos(2 * np.pi * time_features['hour'] / 24.0)
time_features['month_sin'] = np.sin(2 * np.pi * time_features['month'] / 12.0)
time_features['month_cos'] = np.cos(2 * np.pi * time_features['month'] / 12.0)

# We only add sin/cos columns to the main data
data = pd.concat([data, time_features[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']]], axis=1)

# 4. Scaling
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

# 5. Sliding Window
def create_daily_sequences(data_array, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data_array) - lookback - horizon + 1):
        X.append(data_array[i : (i + lookback), :])
        y.append(data_array[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X, y = create_daily_sequences(data_scaled, 48, 24)

# 6. Data Split (Train / Val / Test)
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = torch.from_numpy(X[:train_size]).float(), torch.from_numpy(y[:train_size]).float()
X_val, y_val = torch.from_numpy(X[train_size : train_size + val_size]).float(), torch.from_numpy(y[train_size : train_size + val_size]).float()
X_test, y_test = torch.from_numpy(X[train_size + val_size:]).float(), torch.from_numpy(y[train_size + val_size:]).float()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

# ===========================================
# 7. ARCHITECTURAL DEFINITIONS (Input size is 7)
# ===========================================
# 3 (Sun, Wind, Consumption) + 4 (Time Codings) = 7 Variables
INPUT_SIZE = 7
OUTPUT_SIZE = 7 * 24 # Tüm 7 değişkenin 24 saatlik tahmini

class GRU_Model(nn.Module):
    def __init__(self):
        super(GRU_Model, self).__init__()
        self.gru = nn.GRU(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class LSTM_Model(nn.Module):
    def __init__(self):
        super(LSTM_Model, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class BiLSTM_Model(nn.Module):
    def __init__(self):
        super(BiLSTM_Model, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(128, OUTPUT_SIZE) 

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# ===========================================
# 8. JOINT TRAINING AND TESTING FUNCTION
# ===========================================
def train_and_evaluate(model, model_name, lr=0.001):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience = 5
    epochs_no_improve = 0
    
    print(f"\n--- {model_name} EĞİTİMİ BAŞLADI (Zaman Kodlamalı) ---")
    for epoch in range(25):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f"{model_name}: Erken Durdurma Tetiklendi (Epoch {epoch+1})")
                break
                
    # Load up and test the best weights
    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).cpu().numpy()
        
    # Reverse scaling process (we do dynamic reshaping since there are 7 variables)
    preds_mw = scaler.inverse_transform(preds.reshape(-1, INPUT_SIZE)).reshape(preds.shape)
    y_test_mw = scaler.inverse_transform(y_test.numpy().reshape(-1, INPUT_SIZE)).reshape(y_test.shape)

    # We draw Index 2 (Load - Consumption) values
    preds_load = preds_mw[:, 2::INPUT_SIZE] 
    y_test_load = y_test_mw[:, 2::INPUT_SIZE]

    mae = mean_absolute_error(y_test_load, preds_load)
    wape = (mae / np.mean(y_test_load)) * 100
    
    print(f"[{model_name}] Başarıyla Tamamlandı -> MAE: {mae:.2f} MW | WAPE: %{wape:.2f}")

# ===========================================
# 9. COACHING AND CREATING A LEADERBOARD
# ===========================================
gru_model = GRU_Model()
lstm_model = LSTM_Model()
bilstm_model = BiLSTM_Model()

train_and_evaluate(gru_model, "GRU")
train_and_evaluate(lstm_model, "LSTM")
train_and_evaluate(bilstm_model, "Bi-LSTM")